In [6]:
# system utilities
import os
import glob
import copy

# to logg messages and debug
import logging

logging.basicConfig(
#    filename="test.log",
    level=logging.DEBUG,
    format="%(asctime)s:%(levelname)s:%(message)s"
    )

# modules to query skymapper 
import urllib.request

# plotting module
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec

# numeric and table data modules
import numpy as np
from numpy import errstate,isneginf,array
from scipy.optimize import fsolve

# dataframes module
import pandas as pd
pd.set_option('display.max_columns', None)
pd.set_option('display.max_rows', None)

# module to read Fits files
from astropy.io import fits

# image visualization and World Coordinate System modules
from astropy.visualization import simple_norm
from astropy.wcs import WCS

# units and coordinate handling
from astropy import units as u
from astropy.coordinates import SkyCoord

# Apperture photometry module
from photutils import SkyCircularAperture, aperture_photometry




In [7]:
def getKeyListVal(filename, keys):
    keys_dict = {}
    with fits.open(filename) as hdu_list:
        primary_header = hdu_list[0].header
        for key in keys:
            keys_dict[key] = primary_header[key]
        return keys_dict


# Only skymapper defined and only for r_SDSS and NB_659
# but it should be easily extensible
#
class ZPCalibration:    
    """A class to contain a pandas dataframe with the data
    to calibrate frame and the zero points needed for the calibration."""
    def __init__(self, frame, calibrator, mlim, mag_min, mag_max, dmag_filter, ap_radius):
        self.frame = frame
        self.basedir = os.path.dirname(frame)
        logging.info('ZPCalibration: basedir: %s'%self.basedir)
        
        # some variables not yet determined
        self.aperture = None
        self.data = None
        self.hdr = None
        self.wcs = None
        self.convFactor_ADU_Flambda = None
        
        # get sky levels from orig files
        # these levels were substracted by swarp
        self.orig_skies_adu = self.getOrigSkyLevels()
        
        # information needed to do the calibration
        self.calibrator = calibrator
        self.mlim = mlim
        self.mag_min =  mag_min
        self.mag_max = mag_max
        self.dmag_filter = dmag_filter
        self.band = getKeyListVal(frame, ['FILTER'])['FILTER'].strip() 

        # we load the df with the calibrator data
        # we need positions and magnitudes
        #
        if calibrator == 'skymapper':
            self.df = self.getSkyMapperDf()
        else:
            self.df = None
            
        # Now we load the dataframe with the raw magnitudes in band self.band
        # Note that self.band will be the name of the column of the data frame
        
        # we add the raw magnitudes to self.df
        self.ap_radius = ap_radius
        self.doPhot()
        
        # here we get the zero point
        self.zp, self.slope, self.sigma = self.getZP()
        
        # we get the conversion factor to Flambda in erg/s/cm^2
        # imaging data provides physical units of erg/s/cm^2 therefore
        # we need to multiply by filter width
        #
        self.zp2 = self.getFlambdaZP(self.getPivotWavelength(self.band))
        self.convFactor_ADU_Flambda = 10**((self.zp2 + self.zp)/2.5)
        self.filterWidth = self.getFilterWidth(self.band)
        
        # get sky levels in physical units. Note I multiplied by filterWidth (not so the DNs)
        # so the units will be erg/s/cm**2
        #
        self.orig_skies_phys = copy.deepcopy(self.orig_skies_adu)
        self.orig_skies_phys.update((x, y*self.convFactor_ADU_Flambda*self.filterWidth) for x, y in self.orig_skies_phys.items())

        sky_mean_list = []
        sky_std_list = []
        for key in self.orig_skies_phys:
            sky_mean_list.append(self.orig_skies_phys[key][0])
            sky_std_list.append(self.orig_skies_phys[key][1])
        self.mean_sky = np.mean(sky_mean_list)
        self.std_sky = np.mean(sky_std_list)
        
        # the DN units once we multiply by self.convFactor_ADU_Flambda
        # These are the units required to convert to AB magnitudes by simply 
        # integrating over circular area around stars.
        # It also gives the proper mags/arcsec*2 values by integarting
        # over sky area.
        # But proper imaging fluxes needs erg/s/cm**2
        #
        self.bunit = 'erg/s/cm**2/Angstrom' # The DN will be in these units.
        
        # here we add std_mags column  to self.df
        self.doPhot() # at this point it will do the photometry on physical values
        
        # create dictionary with keywords to be saved
        self.keyword_dict = {}
        self.keywords, self.comment = self.getKeywords()
        

    def getOrigSkyLevels(self):
        """Read original files used by Swarp to read the sky levels in ADUs."""
        logging.info('getOrigSkyLevels: reading original files used by Swarp to determine sky levels.')
        ext_name_list = ["ESO_CCD_#%2d"%(i) for i in range(65,97)]

        list_to_process = glob.glob(self.basedir+'/OMEGA.*_pre/OMEGA.*_pre_wcs.fits')

        orig_skies_adu = {}
        if len(list_to_process) > 0:
            i = 0
            for file_name in list_to_process:
                i += 1
                if i <= len(list_to_process):
                    base_name = os.path.basename(file_name)
                    hdu_list = fits.open(file_name)
                    
                    median_list = []
                    for extname in ext_name_list:
                        hdr = fits.getheader(file_name, extname=extname)
                        data = fits.getdata(file_name, extname=extname)
                        median_list.append(np.nanmedian(data))
                    median_list = np.array(median_list)            
                    sky_mean = np.nanmean(median_list)
            
                    sky_std = np.nanstd(median_list)            
                    orig_skies_adu[base_name] = np.array([sky_mean, sky_std])
                    logging.info('getOrigSkyLevels: sky %i/%i: %s %5.1f +- %5.1f' % 
                          (i, len(list_to_process), base_name, \
                              orig_skies_adu[base_name][0], orig_skies_adu[base_name][1],))
            
                    #hdu_list.info()
        logging.info('Done.')
        return orig_skies_adu
    
    def getSkyMapperDf(self):
        """Read tke skymapper catalogue to get magnitudes of star in frame."""
        logging.info('getSkyMapper: querying skymapper and creating pandas data frame with magnitudes.')

        coord = getKeyListVal(self.frame, ['CRVAL1', 'CRVAL2'])
        ra_str = ("%-10.5f" % coord['CRVAL1']).strip()
        de_str = ("%-10.5f" % coord['CRVAL2']).strip()
        radius = '0.7'
        self.query_str = "http://skymapper.anu.edu.au/sm-cone/public/query?RA="+ra_str+"&DEC="+de_str+"&SR=0.80&VERB=3&RESPONSEFORMAT=CSV"
        logging.debug(self.query_str)
        with urllib.request.urlopen(self.query_str) as f:
            charset = f.headers.get_content_charset()
            # print(charset)
            df_cat = pd.DataFrame([x.split(',') for x in f.read().decode(charset).split('\r\n')])

        # we set column 1 as the header collumn
        df_cat.columns = df_cat.iloc[0]

        # we select only relevant collumns -- FILTER DEPENDENT
        if self.band == 'r_SDSS' or self.band == 'NB_659':
            df_band_columns = ['g_psf', 'e_g_psf', 'r_psf', 'e_r_psf']
            df_pos_columns = ['raj2000', 'dej2000']
            df_total_columns = ['object_id']+df_pos_columns+['class_star'] + df_band_columns
            df_cat = df_cat[df_total_columns][1:]
            
            # update Class accordingly
            self.ref_band_name = 'sm_r_SDSS'
            self.ref_sideband_name = 'sm_g_SDSS'

        # we replace empty values by nan
        for column in df_pos_columns+df_band_columns:
            df_cat[column]=df_cat[column].replace('',np.nan).astype(float)
            df_cat[column] = df_cat[column].astype('float64')        

        # we select according to mlim and columns with valid coords
        bool_select = (df_cat['r_psf'] < mlim) & \
            (df_cat['g_psf'] < mlim) & \
            (df_cat['raj2000'] == df_cat['raj2000']) & \
            (df_cat['dej2000'] == df_cat['dej2000'])
        df_cat = df_cat[bool_select]   

        # we rename collumns to have a uniform interface for all calibrators
        df_cat.rename(columns={'raj2000': 'ra', 'dej2000': 'dec', \
                     'r_psf': 'ref_band', 'e_r_psf': 'ref_band_e', \
                    'g_psf': 'ref_sideband', 'e_g_psf': 'ref_sideband_e'}, inplace=True)

    
        return df_cat
        

    def doPhot(self):
        """Doing aperture photometry using positions of stars from sky mapper.
           It sets the following class variables:
            - self.aperture
            - self.data
            - self.hdr
            - self.wcs
            - Add column raw_mags to self.df"""
        logging.info('doPhot: aperture photometry on frame.')
    
        if self.aperture is None:
            ra = np.array(self.df['ra'])*u.deg
            dec = np.array(self.df['dec'])*u.deg

            # Define positions and apertures
            logging.info('doPhot: creating aperture object')
            positions = SkyCoord(ra=ra, dec=dec,frame='icrs')
            self.aperture = SkyCircularAperture(positions, r=self.ap_radius * u.arcsec)

        if self.data is None:
            # open image and read
            logging.info('doPhot: reading frame')
            self.data, self.hdu = fits.getdata(self.frame, header=True)
        if self.wcs is None:
            # read wcs from image keywords
            logging.info('doPhot: loading wcs info from frame')
            self.wcs = WCS(self.frame)  

        if self.convFactor_ADU_Flambda is None:
            # we do the aperture photometry -- first good enough version.
            # For a better approach we should do curve of growth..
            #
            logging.info('doPhot: doing aperture photometry on frame.')
            phot_table = aperture_photometry(self.data, self.aperture, wcs=self.wcs)
            phot_table['aperture_sum'].info.format = '%.8g'  # for consistent table output

            # we read the magnitudes from the tables ocam_band
            # df1 = df1.assign(e=pd.Series(np.random.randn(sLength)).values)
            logging.info('doPhot: Add column with raw magnitudes to data frame.')
            self.df = self.df.assign(raw_mags=pd.Series(-2.5*np.log10(np.array(phot_table['aperture_sum']))).values)
            logging.info('doPhot: Done.')
        
        else: # we have convertion to physical units, so we proceed in a slightly modified way
            logging.info('doPhot: photometry on frame with pixel values in physical magnitudes.')

            phot_table = aperture_photometry(self.data*self.convFactor_ADU_Flambda, self.aperture, wcs=self.wcs)
            phot_table['aperture_sum'].info.format = '%.8g'  # for consistent table output
            ocam_band = -2.5*np.log10(np.array(phot_table['aperture_sum'])) + self.getPhysicalUnitsZP()

            # we add column with standard magnitude to self.df
            logging.info('doPhot: adding column with std_mags to data frame.')
            self.df = self.df.assign(std_mags=pd.Series(ocam_band).values)
        # display(self.df)
  
    

    def getZP(self):
        """Calculates the zero point using the reference catalogue."""
        logging.info('getZP: calculating zero points.')
        ref_band = np.array(self.df['ref_band'])
        ref_sideband = np.array(self.df['ref_sideband'])
        ocam_band = np.array(self.df['raw_mags'])
        
        X = []
        Xg = []
        Y = []
        for i in range(len(ref_band)):
            if (ref_band[i] == ref_band[i]) and \
                not (np.isinf(ocam_band[i]) or np.isnan(ocam_band[i])) and \
                (ref_band[i] > self.mag_min and ref_band[i] < self.mag_max):
                X.append(ref_band[i])
                Xg.append(ref_sideband[i])
                Y.append(ocam_band[i])

        X = np.array(X)
        Xg = np.array(Xg)
        Y = np.array(Y)
        
        # needs to modify this to force slope=1.00
        
        s = 1.0000
        c = fsolve(lambda i: ((Y - (s*X + i))**2).sum(), 0.) # ((Y - (s*X + i))**2).sum()
        model = (s, c[0])

        # we do one iteration of sigma clipping
        X1 = []
        Xg1 = []
        Y1 = []
        for i in range(len(X)):
            ypred = model[0]*X[i] + model[1]
            if np.abs(Y[i] - ypred) < self.dmag_filter:
                X1.append(X[i])
                Xg1.append(Xg[i])
                Y1.append(Y[i])

        X1 = np.array(X1)
        Xg1 = np.array(Xg1)
        Y1 = np.array(Y1)

        # we fix slope to be 1.000 need to add colour term...
        s = 1.0000
        c = fsolve(lambda i: ((Y1 - (s*X1 + i))**2).sum(), c[0]) # ((Y - (s*X + i))**2).sum()
        model = (s, c[0])
        
        logging.debug('getZP: model: %8.4f %8.4f'%(model[0], model[1],))
        #model = np.polyfit(X1, Y1, 1)
    
        # with respect to this model
        ocam_prediced_mag = []
        for i in range(len(ref_band)):
            ocam_prediced_mag = np.append(ocam_prediced_mag, model[0]*ref_band[i] + model[1])
    
        # filter nan and inf values
        good_values1 = ~np.isnan(ocam_band)
        good_values2 = ~np.isinf(ocam_band)
        good_values = good_values1 & good_values2
    
        sigma = np.std(ocam_band[good_values] - ocam_prediced_mag[good_values])
    
        slope = model[0]
        zp = model[1]
        return zp, slope, sigma
    
    def getFlambdaZP(self, lam):
        """Calculates zero point to go from physical flux to AB magnitudes."""
        logging.info('getFlambdaZP: get zero point to go from physical flux to AB magnitudes.')
        zp = -2.5*np.log10(3.34*10**(-19)) -5.0*np.log10(lam) - 48.60
        return zp

    # Girardi 2004: https://www.aanda.org/articles/aa/full/2004/28/aa0250-04/aa0250-04.html
    # SSDS DR1: http://classic.sdss.org/dr1/algorithms/fluxcal.html
    # 

    def getFilterWidth(self, band):
        """Filter widths for OCAM band passes"""
        pivWave = {'u_SDSS': 560.0,
              'g_SDSS': 1340.0,
              'r_SDSS': 1350.0,
              'NB_659':  105.0, # This is the VPHAS+ filter
              'i_SDSS': 1550.0,
              'z_SDSS':  800.0,
              'B_JOHN': 1030.0, # This is the Bessell B isophotal mag
              'V_JOHN':  980.0} # This is the Bessel V isophotal mag
    
        if band in pivWave.keys():
            return pivWave[band]
        else:
            raise Exception('getFilterWidth: unrecognize filter pass band %s'%(band))        
        
    def getAB_delta(self, band):
        """Delta magnitude from Vega to AB system."""
        abDelta = {'u_SDSS': 0.0,
              'g_SDSS': 0.0,
              'r_SDSS': 0.0,
              'NB_659': 0.0,
              'i_SDSS': 0.0,
              'z_SDSS': 0.0,
              'B_JOHN': 0.04, # This is the Bessell B isophotal mag
              'V_JOHN': 0.0} # This is the Bessel V isophotal mag
    
        if band in abDelta.keys():
            return abDelta[band]
        else:
            raise Exception('getAB_delta: unrecognize filter pass band %s'%(band))
    
    def getPivotWavelength(self, band):
        """Returns the pivot wavelength needed in the
        conversion from AB magnitudes to physical flux."""
        pivWave = {'u_SDSS': 3557.0,
              'g_SDSS': 4702.0,
              'r_SDSS': 6175.0,
              'NB_659': 6590.0,
              'i_SDSS': 7491.0,
              'z_SDSS': 8946.0,
              'B_JOHN': 4371.0, # This is the Bessell B isophotal mag
              'V_JOHN': 5478.0} # This is the Bessel V isophotal mag
    
        if band in pivWave.keys():
            return pivWave[band]
        else:
            raise Exception('getPivotWavelength: unrecognize filter pass band %s'%(band))
    
    # Get the zero point when DN are in physical unitsç
    # of erg/s/cm^-2/Angstrom
    def getPhysicalUnitsZP(self):
        """Returns the zero point to add to -2.5log(flux)
        to get the AB magnitudes in band self.band"""
        return -2.5*np.log10(3.34*10**(-19)) - 5.0*np.log10(self.getPivotWavelength(self.band)) - 48.60
 
        
    def getKeywords(self):
        """Load dictionary with keywords to be writen to primary headers"""
        logging.info('getKeywords: ')

        # the keys are the keywords. The values the corresponding dictionary values.
        keyword = {}
        comment = {}
        
        # we create PRO keywords to indicate frames that went into coadd.fits
        ancestor_frames_keys = ['HIERARCH MEFPP PRO ANCESTOR{:02d}'.format(i+1) for i in range(len(self.orig_skies_phys))]
        sky_mean_keys = ['HIERARCH MEFPP PRO ANCESTOR{:02d} SKYMEAN'.format(i+1) for i in range(len(self.orig_skies_phys))]
        sky_std_keys = ['HIERARCH MEFPP PRO ANCESTOR{:02d} SKYSTD'.format(i+1) for i in range(len(self.orig_skies_phys))]

        i=0
        zkeys = self.orig_skies_phys.keys()
        for akey in self.orig_skies_phys.keys():
            logging.info('getKeywords: '+akey)
            logging.info('getKeywords: '+ancestor_frames_keys[i])
            
            # frames 
            keyword[ancestor_frames_keys[i]] = akey
            comment[ancestor_frames_keys[i]] = 'Input to Swarp %d' % (i+1)
            logging.info('getKeywords: '+ancestor_frames_keys[i]+' = '+keyword[ancestor_frames_keys[i]]+ \
                                                    ' / '+comment[ancestor_frames_keys[i]])
            
            keyword[sky_mean_keys[i]] = self.orig_skies_phys[akey][0]
            comment[sky_mean_keys[i]] = 'Mean of sky %d' % (i+1)
            logging.info('getKeywords: '+sky_mean_keys[i]+' = '+'%.3E'%(keyword[sky_mean_keys[i]]))
                         
            keyword[sky_std_keys[i]] = self.orig_skies_phys[akey][1]
            comment[sky_std_keys[i]] = 'Std dev of sky %d' % (i+1)
            logging.info('getKeywords: '+sky_std_keys[i]+' = '+'%.3E'%(keyword[sky_std_keys[i]]))
            i+=1

        # we also report the units
        units_sky_key = 'HIERARCH MEFPP PRO SKY UNITS'
        units_sky_val = 'erg/s/cm**2'
        units_sky_comment = 'Units of sky mean and std'
        keyword[units_sky_key] = units_sky_val
        comment[units_sky_key] = units_sky_comment
        logging.info('getKeywords: '+units_sky_key+' = '+keyword[units_sky_key]+' / '+comment[units_sky_key])

        # we report the overall sky values and 1 sigma (these are the important values...)
        mean_sky_qc_key = 'HIERARCH MEFPP QC SKYMEAN'
        keyword[mean_sky_qc_key] = self.mean_sky
        comment[mean_sky_qc_key] = 'Mean sky value in physical units'
        logging.info('getKeywords: '+mean_sky_qc_key +' = '+'%.3E'%(keyword[mean_sky_qc_key] )+' / '+comment[mean_sky_qc_key])
        
        units_skymean_qc_key = 'HIERARCH MEFPP QC SKYMEAN UNITS'
        units_skymean_qc_val = 'erg/s/cm**2'
        keyword[units_skymean_qc_key] = units_skymean_qc_val
        comment[units_skymean_qc_key] = 'Units for QC SKYMEAN'
        logging.info('getKeywords: '+units_skymean_qc_key+' = '+keyword[units_skymean_qc_key]+' / '+comment[units_skymean_qc_key])
        
        std_sky_qc_key = 'HIERARCH MEFPP QC SKYSTD'
        keyword[std_sky_qc_key] = self.std_sky
        comment[std_sky_qc_key] = 'Std dev of sky in physical units'        
        logging.info('getKeywords: '+std_sky_qc_key +' = '+'%.3E'%(keyword[std_sky_qc_key] )+' / '+comment[std_sky_qc_key])
        
        units_skystd_qc_key = 'HIERARCH MEFPP QC SKYSTD UNITS'
        units_skystd_qc_val = 'erg/s/cm**2'
        keyword[units_skystd_qc_key] = units_skystd_qc_val
        comment[units_skystd_qc_key] = 'Units for QC SKYSTD'
        logging.info('getKeywords: '+units_skystd_qc_key+' = '+keyword[units_skystd_qc_key]+' / '+comment[units_skystd_qc_key])
        
        # we report the units of the DN
        keyword['BUNIT'] = self.bunit
        comment['BUNIT'] = 'Units of Data Numbers'
        return keyword, comment

In [ ]:
#########-#########-#########-#########-#########-#########-#########-#########+
# Convert coadd.fits to physical units:
#
# + load swarp output frame, coadd.fits, and read ra, dec
# + load original reduced frames and calculate sky levels
# + load reference catalogue for that ra, dec(only skymapper for now)
#   into a data frame
# + get positions of stars in reference catalogues
# + do aperture photometry in frame using reference catalogue positions
# + load raw magnitudes to data frame
# + calculate conversion factor to standard system
# + calculate conversion factor to physical units
# + calculate sky level in physical units
# - create dictionary with keywords to be writen to coadd.fits
# - multiply coadd.fits by conversion factor and write keywords
# - if desired reload the frame, do the photometry and check
# - write dataframe to basedir
#
##############################################################################

reduced_dir = '/home/fselman/omegacam/fernando/reduced/science/J1649-3307_164955p600_m330702p0/r_SDSS/2017-04-13T05:11:38/'
current_dir = '/home/fselman/omegacam/fernando/'

frame = reduced_dir+'coadd.fits'

mlim = 18.0
mag_min = 13.5
mag_max = 17.0
dmag_filter = 0.2
aperture_radius = 4.0 # arcsec

zp_cal = ZPCalibration(frame, 'skymapper', mlim, mag_min, mag_max, dmag_filter, aperture_radius)


2020-05-06 20:36:05,685:INFO:ZPCalibration: basedir: /home/fselman/omegacam/fernando/reduced/science/J1649-3307_164955p600_m330702p0/r_SDSS/2017-04-13T05:11:38
2020-05-06 20:36:05,686:INFO:getOrigSkyLevels: reading original files used by Swarp to determine sky levels.
2020-05-06 20:36:14,337:INFO:getOrigSkyLevels: sky 1/5: OMEGA.2017-04-13T06:07:44.361_pre_wcs.fits 774.5 +-  76.6
2020-05-06 20:36:31,282:INFO:getOrigSkyLevels: sky 2/5: OMEGA.2017-04-13T06:14:00.502_pre_wcs.fits 787.8 +-  98.0
2020-05-06 20:36:36,546:INFO:getOrigSkyLevels: sky 3/5: OMEGA.2017-04-13T06:09:49.512_pre_wcs.fits 778.5 +-  85.7
2020-05-06 20:36:52,366:INFO:getOrigSkyLevels: sky 4/5: OMEGA.2017-04-13T06:05:39.010_pre_wcs.fits 765.3 +-  66.6
2020-05-06 20:37:04,743:INFO:getOrigSkyLevels: sky 5/5: OMEGA.2017-04-13T06:11:54.501_pre_wcs.fits 787.2 +-  93.6
2020-05-06 20:37:04,744:INFO:Done.
2020-05-06 20:37:04,755:INFO:getSkyMapper: querying skymapper and creating pandas data frame with magnitudes.
2020-05-06 20:37

In [ ]:
# we plot the magnitudes from the photometry table
ref_band = np.array(zp_cal.df['ref_band'])
ref_sideband = np.array(zp_cal.df['ref_sideband'])
ocam_band = np.array(zp_cal.df['std_mags'])

data = zp_cal.data
wcs = zp_cal.wcs

sigma = zp_cal.sigma
mag_max = zp_cal.mag_max
mag_min = zp_cal.mag_min

# we plot the ocam vs skymapper relation
# figsize=(8, 4)
fig = plt.figure(figsize=(14.26, 11.0))

gs = gridspec.GridSpec(nrows=4, ncols=5, wspace=0.0) # width_ratios=[1,1,1,1,1])
ax1 = fig.add_subplot(gs[:-2,:-3])
ax2 = fig.add_subplot(gs[-2, :-3], sharex=ax1)
ax3 = fig.add_subplot(gs[:-1,  -3:])
ax4 = fig.add_subplot(gs[-1,:])


#plt.figure(figsize = (10,10))
#ax1.set_aspect('equal')
ax1.scatter(ref_band, ocam_band,  color='black', s=2)
ax1.set_xlim(8.0, mag_max+1.0)
ax1.set_ylim(8.0, mag_max+1.0)

# delta zp vs mag
ax2.scatter(ref_band, ocam_band - ref_band,  color='black', s=2)
ax2.set_ylim(-10.0*sigma, 10.0*sigma)
ax2.axhline(0.0, ls=':')
ax2.axvline(mag_min, ls=':')
ax2.axvline(mag_max, ls=':')

# and we plot apertures on image
  
norm = simple_norm(data, 'sqrt', percent=99)
ax3.imshow(data, norm=norm)
zp_cal.aperture.to_pixel(wcs).plot(color='white', lw=1)

plt.setp(ax3.get_xticklabels(), visible=False)
plt.setp(ax3.get_yticklabels(), visible=False)

# delta zp vs color
color = ref_sideband - ref_band
select = ref_band > mag_min

# the slope of 0.994 gives a bias at fainter magnitudes
# that translate into an offset. If we set slope to 1
# for this plot we get an offset 0.1 mags.
#
ax4.scatter(color[select], ocam_band[select] - ref_band[select],  color='black', s=2)
ax4.set_ylim(-3.0*sigma, 3.0*sigma)
ax4.set_xlim(-0.5, 2.5)
ax4.axhline(0.0, ls=':')
# plt.setp([ax1, ax2, ax3, ax4], title='ZP Conversion')

plt.show()


In [ ]:
# Plot zeropoint versus magnitude
x = ref_band
xg = ref_sideband
y = ocam_band

plt.figure(figsize = (10,3))
#plt.xlim(8.5, 17.0)
plt.ylim(-0.2, 0.2)
plt.scatter(xg - x, y - x,  color='black', s=2)
plt.title('OCAM ZP')
plt.xlabel('SkyMapper (g-r)_SDSS')
plt.ylabel('OCAM - SkyMapper r_SDSS')
plt.axhline(y=0.0)
#plt.xticks(())
#plt.yticks(())

#plt.plot(X1, Y1 - (model[0]*X1+model[1]), color='red',linewidth=2)
plt.show()

# Plot zeropoint versus color
plt.figure(figsize = (10,3))
plt.xlim(8.5, 17.0)
plt.ylim(-0.2, 0.2)
plt.scatter(x, y - x,  color='black', s=2)
plt.title('OCAM ZP')
plt.xlabel('SkyMapper r_SDSS')
plt.ylabel('OCAM - SkyMapper r_SDSS')
plt.axhline(y=0.0)
#plt.xticks(())
#plt.yticks(())

plt.show()
